# RNA — Imitación de jugador experto vs. Lace 1 (Hollow Knight: Silksong)

Proyecto personal de portfolio, materia Inteligencia Artificial (UTN FRBA). Clasificación
supervisada (no RL): dado el estado del juego en un instante, predecir la acción que tomaría
un jugador experto (yo) durante el combate contra Lace 1.

Estructura y convenciones de esta notebook alineadas a los demos de la cátedra
(`RNA-MLP-procesar-datos.ipynb`, `ConvNet-procesar-datos.ipynb`): Keras (TensorFlow), split con
`scikit-learn`, matriz de confusión y `classification_report` por clase.

**Diferencia clave con los demos de clase**: acá el atributo clase no es único — son 5 salidas
independientes (`move`, `look`, `jump`, `attack`, `dash`), porque el jugador puede combinarlas
libremente en un mismo instante. Se implementa con la API funcional de Keras: un tronco de capas
compartidas + 5 cabezas de salida, cada una con su propia función de pérdida.

**Nota**: no hace falta GPU para este dataset (tabular, volumen chico) — corre bien en CPU. Si
usás Colab, el runtime "CPU" alcanza; si corrés localmente con GPU AMD, tampoco hay que
configurar nada especial, TensorFlow en CPU es más que suficiente acá.

In [ ]:
#@title Librerías a usar
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model, to_categorical

import tensorflow as tf
from matplotlib import pyplot as plt
import pandas as pd
import numpy as np
import math
import os

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

print("Librerías cargadas. TensorFlow", tf.__version__)


# Datos

In [ ]:
#@title Obtener el CSV

#@markdown - "Subir archivo": abre el selector de Colab, subís el CSV directo desde tu PC al
#@markdown   entorno de ejecución (no se guarda en Drive, desaparece si se reinicia el runtime).
#@markdown - "Google Drive": monta tu Drive y lee desde ahí (persiste entre sesiones).
#@markdown - "Ruta local": para correr esta notebook fuera de Colab (VS Code / Jupyter local).
origen_datos = "Subir archivo"  #@param ["Subir archivo", "Google Drive", "Ruta local"]

path = None
archivo_subido = None

if origen_datos == "Subir archivo":
    from google.colab import files
    subidos = files.upload()  # abre el diálogo; elegí el CSV
    archivo_subido = list(subidos.keys())[0]
    print(f"Subido: {archivo_subido}")
elif origen_datos == "Google Drive":
    from google.colab import drive
    drive.mount('/content/gdrive', force_remount=True)
    path = '/content/gdrive/MyDrive/SilksongRL_Dataset/'  #@param {type:"string"}
else:
    path = './SilksongRL_Dataset/'  #@param {type:"string"}


In [ ]:
#@title Cargar datos

archivo_datos = 'lace1_dataset.csv'  #@param {type:"string"}

if archivo_subido is not None:
    ruta_completa = archivo_subido  # ya está en el working dir de Colab tras el upload
else:
    ruta_completa = os.path.join(path, archivo_datos)

assert os.path.isfile(ruta_completa), f"No se encontró el archivo: {ruta_completa}"

df = pd.read_csv(ruta_completa)
print(f"Filas cargadas: {len(df)}")
print(f"Episodios (intentos) distintos: {df['episode_id'].nunique()}")
df.head()


In [ ]:
#@title Definir columnas de entrada y de salida

# 21 features de estado, ya normalizadas a [0,1] por LaceEncounter.cs — no hace falta
# aplicar StandardScaler/MinMaxScaler acá (a diferencia del demo de clase).
STATE_COLS = [
    'hero_x', 'hero_y', 'hero_vel_x', 'hero_vel_y', 'hero_hp',
    'boss_x', 'boss_y', 'boss_vel_x', 'boss_vel_y', 'boss_hp',
    'attack_idle', 'attack_comboslash', 'attack_counter', 'attack_rapidslash',
    'attack_jslash', 'attack_downstab', 'attack_charge', 'attack_evade',
    'attack_crossslash', 'attack_stun', 'attack_multihit',
]

EPISODE_COL = 'episode_id'
TICK_COL = 'tick'
OUTCOME_COL = 'outcome'

# Las 5 cabezas de salida: nombre de columna, tipo (categórica N clases / binaria) y clases
ACTION_HEADS = {
    'move':   {'col': 'move',   'type': 'categorical', 'n_classes': 3, 'labels': ['None', 'Left', 'Right']},
    'look':   {'col': 'look',   'type': 'categorical', 'n_classes': 3, 'labels': ['None', 'Up', 'Down']},
    'jump':   {'col': 'jump',   'type': 'binary',       'n_classes': 2, 'labels': ['No', 'Sí']},
    'attack': {'col': 'attack', 'type': 'binary',       'n_classes': 2, 'labels': ['No', 'Sí']},
    'dash':   {'col': 'dash',   'type': 'binary',       'n_classes': 2, 'labels': ['No', 'Sí']},
}

assert set(STATE_COLS + [EPISODE_COL, TICK_COL, OUTCOME_COL] + [h['col'] for h in ACTION_HEADS.values()]) <= set(df.columns), \
    "Faltan columnas esperadas en el CSV — revisar el header contra DatasetLogger.cs"

for name, h in ACTION_HEADS.items():
    print(f"{name}: distribución de clases")
    print(df[h['col']].value_counts(normalize=True).sort_index())
    print()


# Split train/test (agrupado por episodio)

**Importante**: filas consecutivas del mismo intento (`episode_id`) están muy correlacionadas
temporalmente — no es un split i.i.d. como el de IRIS.csv del demo de clase. Si se separa fila
por fila al azar, quedan frames casi idénticos en train y en test (data leakage), y las métricas
quedan infladas de forma artificial. Por eso el split se hace por **episodio completo**: cada
intento entero va entero a train o a test, nunca partido.

In [ ]:
#@title Separar en entrenamiento y prueba (agrupado por episodio)

proporcion_porcentaje_datos_prueba = 25  #@param {type:"integer"}
# TP2 exige mínimo 20% de test — se fuerza el piso acá
propTest = max(0.20, proporcion_porcentaje_datos_prueba / 100.0)

gss = GroupShuffleSplit(n_splits=1, test_size=propTest, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df[EPISODE_COL]))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(df_train)} filas, {df_train[EPISODE_COL].nunique()} episodios")
print(f"Test:  {len(df_test)} filas, {df_test[EPISODE_COL].nunique()} episodios ({len(df_test)/len(df):.1%})")


# Preparación de datos para los modelos

In [ ]:
#@title Armar X / Y planos (para los modelos MLP, un frame por muestra)

def armar_xy_plano(d):
    X = d[STATE_COLS].to_numpy(dtype=np.float32)
    y = {}
    for name, h in ACTION_HEADS.items():
        vals = d[h['col']].to_numpy()
        if h['type'] == 'categorical':
            y[name] = to_categorical(vals, num_classes=h['n_classes'])
        else:
            y[name] = vals.astype(np.float32)
    return X, y

X_train, y_train = armar_xy_plano(df_train)
X_test, y_test = armar_xy_plano(df_test)

print("X_train:", X_train.shape)
for k, v in y_train.items():
    print(f"  y_train[{k}]:", v.shape)


In [ ]:
#@title Armar secuencias con ventana temporal (para el ConvNet)

window_size = 8  #@param {type:"integer"}
# Default 8 ticks (~0.8s a 10Hz) — cubre la mayoría de los telegraphs de ataque de Lace 1 sin
# arrastrar demasiada historia vieja. Configurable para comparar otros valores más adelante.

def armar_xy_ventaneado(d, window_size):
    X_list, y_list = [], {name: [] for name in ACTION_HEADS}
    # una ventana por episodio, nunca cruza el límite entre intentos distintos
    for ep_id, grupo in d.groupby(EPISODE_COL):
        grupo = grupo.sort_values(TICK_COL)
        estados = grupo[STATE_COLS].to_numpy(dtype=np.float32)
        acciones = {name: grupo[h['col']].to_numpy() for name, h in ACTION_HEADS.items()}
        n = len(grupo)
        if n < window_size:
            continue
        for i in range(window_size - 1, n):
            X_list.append(estados[i - window_size + 1: i + 1])
            for name, h in ACTION_HEADS.items():
                val = acciones[name][i]
                if h['type'] == 'categorical':
                    y_list[name].append(to_categorical(val, num_classes=h['n_classes']))
                else:
                    y_list[name].append(float(val))
    X = np.stack(X_list) if X_list else np.empty((0, window_size, len(STATE_COLS)), dtype=np.float32)
    y = {name: np.array(vals) for name, vals in y_list.items()}
    return X, y

X_train_seq, y_train_seq = armar_xy_ventaneado(df_train, window_size)
X_test_seq, y_test_seq = armar_xy_ventaneado(df_test, window_size)

print("X_train_seq:", X_train_seq.shape, " (muestras, ventana, features)")
print("X_test_seq: ", X_test_seq.shape)


# Modelo

In [ ]:
#@title Funciones para construir los modelos (tronco compartido + 5 cabezas)

def agregar_cabezas_salida(x):
    salidas = {}
    losses = {}
    metrics = {}
    for name, h in ACTION_HEADS.items():
        if h['type'] == 'categorical':
            salidas[name] = layers.Dense(h['n_classes'], activation='softmax', name=name)(x)
            losses[name] = 'categorical_crossentropy'
        else:
            salidas[name] = layers.Dense(1, activation='sigmoid', name=name)(x)
            losses[name] = 'binary_crossentropy'
        metrics[name] = ['accuracy']
    return salidas, losses, metrics


def build_mlp_model(input_dim, capas_ocultas, dropout=0.0, nombre='mlp'):
    entrada = layers.Input(shape=(input_dim,), name='estado')
    x = entrada
    for i, n in enumerate(capas_ocultas):
        x = layers.Dense(n, activation='relu', name=f'oculta_{i+1}')(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    salidas, losses, metrics = agregar_cabezas_salida(x)
    modelo = Model(inputs=entrada, outputs=salidas, name=nombre)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=losses, metrics=metrics)
    return modelo


def build_convnet_model(window_size, n_features, filtros=(16, 32), kernel_size=3, pool_size=2,
                         capas_ocultas=(16,), dropout=0.0, nombre='convnet'):
    entrada = layers.Input(shape=(window_size, n_features), name='ventana_estado')
    x = entrada
    for i, f in enumerate(filtros):
        x = layers.Conv1D(f, kernel_size, activation='relu', padding='same', name=f'conv_{i+1}')(x)
        x = layers.MaxPooling1D(pool_size, padding='same', name=f'pool_{i+1}')(x)
    x = layers.Flatten()(x)
    for i, n in enumerate(capas_ocultas):
        x = layers.Dense(n, activation='relu', name=f'oculta_{i+1}')(x)
        if dropout > 0:
            x = layers.Dropout(dropout)(x)
    salidas, losses, metrics = agregar_cabezas_salida(x)
    modelo = Model(inputs=entrada, outputs=salidas, name=nombre)
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001), loss=losses, metrics=metrics)
    return modelo


In [ ]:
#@title Instanciar las 3 arquitecturas a comparar

# 1) MLP baseline — una capa oculta chica
modelo_mlp_a = build_mlp_model(input_dim=len(STATE_COLS), capas_ocultas=[16], nombre='MLP_A_baseline')

# 2) MLP variante — más profunda, con dropout (mismo tipo de arquitectura vista en clase,
#    topología distinta — cuenta como el 3er entrenamiento con arquitectura distinta que pide el TP2)
modelo_mlp_b = build_mlp_model(input_dim=len(STATE_COLS), capas_ocultas=[32, 16], dropout=0.2, nombre='MLP_B_profunda')

# 3) ConvNet 1D sobre la ventana temporal
modelo_convnet = build_convnet_model(window_size=window_size, n_features=len(STATE_COLS),
                                      filtros=(16, 32), capas_ocultas=(16,), dropout=0.2,
                                      nombre='ConvNet_C_ventana')

modelos = {
    'MLP_A_baseline':  {'modelo': modelo_mlp_a,   'X_train': X_train,     'y_train': y_train,     'X_test': X_test,     'y_test': y_test},
    'MLP_B_profunda':  {'modelo': modelo_mlp_b,   'X_train': X_train,     'y_train': y_train,     'X_test': X_test,     'y_test': y_test},
    'ConvNet_C_ventana': {'modelo': modelo_convnet, 'X_train': X_train_seq, 'y_train': y_train_seq, 'X_test': X_test_seq, 'y_test': y_test_seq},
}

for nombre, m in modelos.items():
    print(f"--- {nombre} ---")
    m['modelo'].summary()
    print()


# Entrenamiento

In [ ]:
#@title Entrenar los 3 modelos

cant_epocas_entrenamiento = 150  #@param {type:"integer"}
porcentaje_datos_validacion = 15  #@param {type:"number"}
epocas_paciencia_estabilidad = 20  #@param {type:"integer"}

porcentaje_datos_validacion = min(49.9, max(0.5, porcentaje_datos_validacion)) / 100.0

for nombre, m in modelos.items():
    print(f"\n=== Entrenando {nombre} ===")
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=epocas_paciencia_estabilidad,
        restore_best_weights=True,
    )
    historia = m['modelo'].fit(
        m['X_train'], m['y_train'],
        validation_split=porcentaje_datos_validacion,
        epochs=cant_epocas_entrenamiento,
        callbacks=[early_stop],
        verbose=0,
    )
    m['historia'] = historia
    print(f"  Épocas entrenadas: {len(historia.history['loss'])}")


In [ ]:
#@title Gráficos de entrenamiento (loss total por modelo)

plt.figure(figsize=(12, 6))
for nombre, m in modelos.items():
    plt.plot(m['historia'].history['loss'], label=f'{nombre} (train)')
    plt.plot(m['historia'].history['val_loss'], '--', label=f'{nombre} (val)')
plt.xlabel('época')
plt.ylabel('loss total (suma de las 5 cabezas)')
plt.title('Curvas de entrenamiento')
plt.legend()
plt.show()


# Evaluación

In [ ]:
#@title Evaluar con datos de prueba — matriz de confusión y precision/recall por cabeza

def evaluar_modelo(nombre, m):
    print(f"\n{'='*60}\n  {nombre}\n{'='*60}")
    pred = m['modelo'].predict(m['X_test'], verbose=0)
    # Keras 3 con salidas nombradas (dict) devuelve un dict {nombre: array} directamente.
    # Con un único output devolvería el array pelado; con lista/tupla, respeta el orden de
    # model.output_names. Estos tres casos cubren cualquier versión/configuración de Keras.
    if isinstance(pred, dict):
        pred_por_cabeza = pred
    elif isinstance(pred, (list, tuple)):
        pred_por_cabeza = dict(zip(m['modelo'].output_names, pred))
    else:
        pred_por_cabeza = {m['modelo'].output_names[0]: pred}

    for name, h in ACTION_HEADS.items():
        y_true_h = m['y_test'][name]
        y_pred_h = pred_por_cabeza[name]
        if h['type'] == 'categorical':
            y_true_cls = np.argmax(y_true_h, axis=1)
            y_pred_cls = np.argmax(y_pred_h, axis=1)
        else:
            y_true_cls = y_true_h.astype(int)
            y_pred_cls = (y_pred_h.reshape(-1) >= 0.5).astype(int)

        print(f"\n--- Cabeza: {name} ---")
        print(classification_report(y_true_cls, y_pred_cls, target_names=h['labels'], zero_division=0))

        cm = confusion_matrix(y_true_cls, y_pred_cls, labels=list(range(h['n_classes'])))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=h['labels'])
        fig, ax = plt.subplots(figsize=(4, 4))
        disp.plot(ax=ax, colorbar=False)
        ax.set_title(f'{nombre} — {name}')
        plt.show()

for nombre, m in modelos.items():
    evaluar_modelo(nombre, m)


### Baseline trivial (clase mayoritaria)

Antes de mirar accuracy/precision de la RNA, conviene saber qué tan fácil es "hacer trampa":
un modelo que siempre predice la clase más frecuente (ej. `move=None`, `dash=No`) puede dar una
accuracy alta sin haber aprendido nada, porque la mayoría de los frames son "sin input". Este
baseline es la vara mínima que cualquier modelo entrenado tiene que superar, sobre todo en
precision/recall de las clases minoritarias — si un modelo entrenado no le gana al baseline en
una cabeza, esa cabeza no está aprendiendo señal útil todavía.

In [ ]:
#@title Calcular baseline de clase mayoritaria (referencia para juzgar si las métricas son buenas)

baseline_resultados = []
for name, h in ACTION_HEADS.items():
    clase_mayoritaria = df_train[h['col']].mode()[0]
    y_true_test = df_test[h['col']].to_numpy()
    y_pred_baseline = np.full_like(y_true_test, clase_mayoritaria)

    reporte = classification_report(y_true_test, y_pred_baseline, labels=list(range(h['n_classes'])),
                                     target_names=h['labels'], zero_division=0, output_dict=True)
    baseline_resultados.append({
        'cabeza': name,
        'clase_mayoritaria': h['labels'][clase_mayoritaria],
        'accuracy_baseline': reporte['accuracy'],
        'macro_f1_baseline': reporte['macro avg']['f1-score'],
    })

df_baseline = pd.DataFrame(baseline_resultados).set_index('cabeza')
print("Comparar esto contra 'Resumen comparativo' de abajo — si un modelo no supera claramente")
print("estos valores (sobre todo el macro_f1, no solo accuracy), no aprendió nada relevante para esa cabeza.")
df_baseline.style.format({'accuracy_baseline': '{:.2%}', 'macro_f1_baseline': '{:.2%}'})


In [ ]:
#@title Resumen comparativo (accuracy por cabeza y por modelo)

resumen = []
for nombre, m in modelos.items():
    pred = m['modelo'].predict(m['X_test'], verbose=0)
    if isinstance(pred, dict):
        pred_por_cabeza = pred
    elif isinstance(pred, (list, tuple)):
        pred_por_cabeza = dict(zip(m['modelo'].output_names, pred))
    else:
        pred_por_cabeza = {m['modelo'].output_names[0]: pred}
    fila = {'modelo': nombre}
    for name, h in ACTION_HEADS.items():
        y_true_h = m['y_test'][name]
        y_pred_h = pred_por_cabeza[name]
        if h['type'] == 'categorical':
            y_true_cls = np.argmax(y_true_h, axis=1)
            y_pred_cls = np.argmax(y_pred_h, axis=1)
        else:
            y_true_cls = y_true_h.astype(int)
            y_pred_cls = (y_pred_h.reshape(-1) >= 0.5).astype(int)
        reporte = classification_report(y_true_cls, y_pred_cls, labels=list(range(h['n_classes'])),
                                         zero_division=0, output_dict=True)
        fila[f'acc_{name}'] = reporte['accuracy']
        fila[f'macroF1_{name}'] = reporte['macro avg']['f1-score']
    resumen.append(fila)

# comparar cada macroF1_* contra df_baseline['macro_f1_baseline'] — ahí se ve si el modelo
# realmente aprendió algo por cabeza, no solo si "acertó mucho" gracias al desbalance de clases
pd.DataFrame(resumen).set_index('modelo').style.format('{:.2%}')
